
### Goals:
- show loss on test examples
- download reconstructed states
- analyze performance lost by trained SAC w/ recon states

In [4]:
import os
import sys
sys.path.append('../')
from pathlib import Path
import torch
import torch.nn.functional as F
import numpy as np
from models.sae import SAE
from scripts.data import DataLoader

Root = Path().resolve().parent

In [5]:
def test(dataset, sae, topk, batch_size=100000):
    size = 100000 - (100000 % batch_size)
    reconstructed = np.array(np.zeros((size, 348)))
    for i in range(size // batch_size):
        batch = dataset.sample(batch_size, dataset="Test")
        y = sae.forward(batch, topk=topk)
        
        reconstructed[i*batch_size:(i+1)*batch_size] = y.detach().numpy()
        
        # compute loss
        recon_loss = F.mse_loss(y, batch)
        l0_norm = (sae.z != 0).float().sum(dim=1).mean().item()

        # print info
        if i % 1 == 0:
            print(f"step: {i+1}, loss: {recon_loss:.02f}, l0 norm: {l0_norm:.1f}")
        # if i % 200000:
        #     print(batch[0] - y[0])
            
    np.save("reconstructed", reconstructed)
    

def load(sae):
    path = Root / "checkpoints/sae"
    model = torch.load(path / "model.pth")
    sae.load_state_dict(model)

In [6]:
sae = SAE(348, 4, 348)
load(sae)
dataset = DataLoader()
test(dataset, sae, topk=True)

step: 1, loss: 13.87, l0 norm: 31.9
